# **1. Coleta de dados**

Este notebook constrói a camada bruta do projeto. O calendário da Série A publicado pela CBF é a base: ele fixa a data, o horário e o estádio oficiais de cada partida de 2020 a 2026. A ESPN fornece o público de cada partida a partir de 2022, e o arquivo histórico do Open-Meteo fornece o clima hora a hora na cidade do estádio.

| etapa | fonte | resultado |
|---|---|---|
| universo de partidas | páginas de temporada e endpoints de rodada da CBF | todas as partidas da Série A de 2020 a 2026 |
| público | API pública de placares da ESPN (`bra.1`) | público, estádio e horário de cada evento |
| cidades-sede | API de geocodificação do Open-Meteo | latitude e longitude de cada cidade-sede |
| clima | arquivo histórico do Open-Meteo | temperatura e precipitação por hora em cada cidade |
| auditoria | arquivos coletados | cobertura por temporada e por fonte |

As temporadas de 2020 e 2021 são coletadas apenas para aquecer o rating Elo e as variáveis de momento; o público é estudado a partir de 2022, a primeira temporada sem limites de capacidade por causa da COVID-19. As respostas originais ficam em `data/raw/`, o que permite reexecutar o notebook sem repetir requisições.

In [1]:
from __future__ import annotations

import gzip
import json
import re
import time
import unicodedata
from datetime import date, datetime, timedelta
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

### **1.1. Configuração**

O período do estudo, o intervalo entre requisições e as pastas de cache são definidos aqui. `REFRESH_CURRENT` força novas requisições para a temporada atual, cujas rodadas ainda estão em andamento.

In [2]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
CBF_CACHE = RAW / "cbf_rounds"
ESPN_CACHE = RAW / "espn_scoreboard"
WEATHER_CACHE = RAW / "open_meteo"
for directory in (RAW, INTERIM, CBF_CACHE, ESPN_CACHE, WEATHER_CACHE):
    directory.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "academic-research/brasileirao-attendance-forecasting"}
CBF_SEASONS = list(range(2020, 2027))
ATTENDANCE_SEASONS = list(range(2022, 2027))
CURRENT_SEASON = 2026
TODAY = date.today()
REQUEST_INTERVAL = 0.35
REQUEST_TIMEOUT = 45
REFRESH_CURRENT = False

CBF_TABLE = "https://www.cbf.com.br/futebol-brasileiro/tabelas/campeonato-brasileiro/serie-a/{year}"
CBF_ROUND = "https://www.cbf.com.br/api/cbf/jogos/campeonato/{competition_id}/rodada/{round}/fase/{phase_id}"
ESPN_SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/soccer/{league}/scoreboard"
GEOCODING = "https://geocoding-api.open-meteo.com/v1/search"
WEATHER_ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"

## **2. Universo de partidas da CBF**

A página de cada temporada expõe os identificadores da competição e da fase; o endpoint de rodada retorna então as 10 partidas de cada uma das 38 rodadas. Cada resposta é salva em JSON compactado.

In [3]:
_last_request_at = 0.0


def get(url: str, params: dict | None = None, as_json: bool = True, attempts: int = 4):
    global _last_request_at
    last_error = None
    for attempt in range(attempts):
        wait = REQUEST_INTERVAL - (time.monotonic() - _last_request_at)
        if wait > 0:
            time.sleep(wait)
        _last_request_at = time.monotonic()
        try:
            response = requests.get(url, params=params, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            return response.json() if as_json else response.text
        except Exception as exc:
            last_error = exc
            time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"Requisição falhou após {attempts} tentativas: {url} {params} {last_error!r}")


def read_cache(path: Path):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        return json.load(handle)


def write_cache(path: Path, payload) -> None:
    with gzip.open(path, "wt", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False)


def cbf_identifiers(year: int) -> tuple[str, str]:
    html = get(CBF_TABLE.format(year=year), as_json=False)
    competition = re.search(r'competitionId\\":\\"(\d+)', html)
    phase = re.search(r'fase_id\\":\\"(\d+)', html)
    if not competition or not phase:
        raise ValueError(f"competitionId ou fase_id não encontrado para {year}")
    return competition.group(1), phase.group(1)


cbf_rounds = {}
for year in tqdm(CBF_SEASONS, desc="Temporadas CBF"):
    paths = [CBF_CACHE / f"{year}_{round_number:02d}.json.gz" for round_number in range(1, 39)]
    refresh = REFRESH_CURRENT and year == CURRENT_SEASON
    if refresh or not all(path.exists() for path in paths):
        competition_id, phase_id = cbf_identifiers(year)
        for round_number, path in zip(range(1, 39), paths):
            if path.exists() and not refresh:
                continue
            url = CBF_ROUND.format(competition_id=competition_id, round=round_number, phase_id=phase_id)
            write_cache(path, get(url))
    for round_number, path in zip(range(1, 39), paths):
        cbf_rounds[(year, round_number)] = read_cache(path)

print(f"respostas de rodada: {len(cbf_rounds):,}")

Temporadas CBF:   0%|          | 0/7 [00:00<?, ?it/s]

respostas de rodada: 266


## **3. Tabela de partidas da CBF**

`cbf_games.csv` mantém uma linha por partida, com data, horário, estádio e placar oficiais. O campo de local segue o padrão `estádio - cidade - estado` e é separado nessas três partes.

In [4]:
def parse_venue(value: str | None) -> tuple[str | None, str | None, str | None]:
    parts = [part.strip() for part in str(value or "").split(" - ")]
    if len(parts) >= 3:
        return " - ".join(parts[:-2]), parts[-2], parts[-1]
    return (parts[0] or None), None, None


def as_int(value):
    try:
        return int(str(value).strip())
    except (TypeError, ValueError):
        return None


rows = []
for (year, round_number), payload in cbf_rounds.items():
    for group in payload.get("jogos", []):
        for game in group.get("jogo", []):
            home, away = game.get("mandante", {}), game.get("visitante", {})
            stadium, city, state = parse_venue(game.get("local"))
            rows.append({
                "game_id": as_int(game.get("id_jogo")),
                "season": year,
                "round": round_number,
                "match_number": as_int(game.get("num_jogo")),
                "date": str(game.get("data") or "").strip() or None,
                "kickoff": str(game.get("hora") or "").strip() or None,
                "venue_raw": game.get("local"),
                "stadium": stadium,
                "city": city,
                "state": state,
                "home_id": as_int(home.get("id")),
                "home_team": home.get("nome"),
                "away_id": as_int(away.get("id")),
                "away_team": away.get("nome"),
                "home_goals": as_int(home.get("gols")),
                "away_goals": as_int(away.get("gols")),
                "has_report": any("mula" in (doc.get("title") or "") for doc in game.get("documentos") or []),
            })

cbf_games = pd.DataFrame(rows).drop_duplicates("game_id").sort_values(["season", "round", "match_number"])
cbf_games["date"] = pd.to_datetime(cbf_games["date"], format="%d/%m/%Y", errors="coerce")
cbf_games.to_csv(INTERIM / "cbf_games.csv", index=False)
display(cbf_games.groupby("season").agg(partidas=("game_id", "nunique"), com_data=("date", "count"), com_sumula=("has_report", "sum")))

,partidas,com_data,com_sumula
season,,,
2020,380,380,380
2021,380,380,380
2022,380,380,380
2023,380,380,380
2024,380,380,380
2025,380,380,380
2026,380,380,360


## **4. Público na ESPN**

A ESPN não aceita intervalos de datas para `bra.1`, então o placar é consultado em cada data em que houve partida da CBF depois da pandemia, mais o dia seguinte, para cobrir jogos noturnos que a ESPN registra na data seguinte. Cada dia é salvo em cache quando já passou.

In [5]:
played = cbf_games[cbf_games["season"].isin(ATTENDANCE_SEASONS) & cbf_games["date"].notna()]
query_dates = sorted({d.date() + timedelta(days=k) for d in played["date"] for k in (0, 1)})
query_dates = [d for d in query_dates if d < TODAY]


def espn_day(day: date, league: str = "bra.1", cache: Path = ESPN_CACHE) -> dict:
    path = cache / f"{day:%Y%m%d}.json.gz"
    if path.exists():
        return read_cache(path)
    payload = get(ESPN_SCOREBOARD.format(league=league), params={"dates": f"{day:%Y%m%d}", "limit": 100})
    if day < TODAY - timedelta(days=2):
        write_cache(path, payload)
    return payload


def espn_events_of(payload: dict) -> list[dict]:
    rows = []
    for event in payload.get("events", []):
        competition = event["competitions"][0]
        sides = {side["homeAway"]: side for side in competition["competitors"]}
        venue = competition.get("venue") or {}
        rows.append({
            "espn_id": int(event["id"]),
            "espn_season": event.get("season", {}).get("year"),
            "kickoff_utc": event.get("date"),
            "status": event.get("status", {}).get("type", {}).get("name"),
            "attendance": competition.get("attendance"),
            "espn_venue": venue.get("fullName"),
            "espn_city": (venue.get("address") or {}).get("city"),
            "espn_home": sides["home"]["team"]["displayName"],
            "espn_home_id": int(sides["home"]["team"]["id"]),
            "espn_away": sides["away"]["team"]["displayName"],
            "espn_away_id": int(sides["away"]["team"]["id"]),
            "espn_home_goals": as_int(sides["home"].get("score")),
            "espn_away_goals": as_int(sides["away"].get("score")),
        })
    return rows


espn_rows = []
for day in tqdm(query_dates, desc="Dias ESPN"):
    espn_rows += espn_events_of(espn_day(day))

espn_events = pd.DataFrame(espn_rows).drop_duplicates("espn_id").sort_values("kickoff_utc")
espn_events.to_csv(INTERIM / "espn_events.csv", index=False)
display(espn_events.groupby("espn_season").agg(eventos=("espn_id", "nunique"), com_publico=("attendance", lambda s: int((s.fillna(0) > 0).sum()))))

Dias ESPN:   0%|          | 0/766 [00:00<?, ?it/s]

,eventos,com_publico
espn_season,,
2022,380,377
2023,380,368
2024,380,376
2025,380,380
2026,363,177


## **5. Coordenadas das cidades-sede**

O clima é tomado na cidade do estádio, precisão suficiente para temperatura e chuva no horário do jogo. Cada par `cidade - estado` é geocodificado pelo Open-Meteo, restrito ao estado correspondente e resolvido para o lugar mais populoso com aquele nome.

In [6]:
STATES = {
    "AC": "Acre", "AL": "Alagoas", "AM": "Amazonas", "AP": "Amapá", "BA": "Bahia", "CE": "Ceará", "DF": "Federal District",
    "ES": "Espírito Santo", "GO": "Goiás", "MA": "Maranhão", "MG": "Minas Gerais", "MS": "Mato Grosso do Sul", "MT": "Mato Grosso",
    "PA": "Pará", "PB": "Paraíba", "PE": "Pernambuco", "PI": "Piauí", "PR": "Paraná", "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte",
    "RO": "Rondônia", "RR": "Roraima", "RS": "Rio Grande do Sul", "SC": "Santa Catarina", "SE": "Sergipe", "SP": "São Paulo", "TO": "Tocantins",
}


def fold(value: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", str(value)) if not unicodedata.combining(c)).casefold().strip()


def geocode(city: str, state: str) -> dict:
    path = WEATHER_CACHE / f"geocode_{fold(city).replace(' ', '_')}_{state}.json"
    if path.exists():
        return json.loads(path.read_text())
    payload = get(GEOCODING, params={"name": city, "count": 20, "language": "en", "countryCode": "BR"})
    state_name = fold(STATES.get(state, ""))
    matches = [r for r in payload.get("results", []) if fold(r.get("admin1", "")).removeprefix("estado de ").removeprefix("estado do ") == state_name]
    matches = sorted(matches, key=lambda r: -(r.get("population") or 0))
    best = matches[0] if matches else {}
    result = {"city": city, "state": state, "latitude": best.get("latitude"), "longitude": best.get("longitude"), "geocoded_name": best.get("name")}
    path.write_text(json.dumps(result, ensure_ascii=False))
    return result


venue_cities = played[["city", "state"]].dropna().drop_duplicates().sort_values(["state", "city"])
cities = pd.DataFrame([geocode(row.city, row.state) for row in venue_cities.itertuples(index=False)])
cities.to_csv(INTERIM / "venue_cities.csv", index=False)
print(f"cidades: {len(cities)} | sem coordenadas: {cities['latitude'].isna().sum()}")
display(cities[cities["latitude"].isna()])

cidades: 26 | sem coordenadas: 0


,city,state,latitude,longitude,geocoded_name


## **6. Clima por hora**

Uma requisição ao arquivo histórico por cidade cobre todo o período depois da pandemia. Os horários vêm no fuso de Brasília, o mesmo do horário de início publicado pela CBF; os dias mais recentes ficam ausentes até o arquivo ser atualizado.

In [7]:
WEATHER_START = date(ATTENDANCE_SEASONS[0], 1, 1)
WEATHER_END = TODAY - timedelta(days=6)


def city_weather(row) -> pd.DataFrame:
    path = WEATHER_CACHE / f"hourly_{fold(row.city).replace(' ', '_')}_{row.state}_{WEATHER_END:%Y%m%d}.json.gz"
    if path.exists():
        payload = read_cache(path)
    else:
        payload = get(WEATHER_ARCHIVE, params={
            "latitude": row.latitude, "longitude": row.longitude,
            "start_date": f"{WEATHER_START}", "end_date": f"{WEATHER_END}",
            "hourly": "temperature_2m,precipitation,relative_humidity_2m",
            "timezone": "America/Sao_Paulo",
        })
        write_cache(path, payload)
    hourly = pd.DataFrame(payload["hourly"])
    return hourly.assign(city=row.city, state=row.state)


weather = pd.concat([city_weather(row) for row in tqdm(list(cities.dropna(subset=["latitude"]).itertuples(index=False)), desc="Clima")], ignore_index=True)
weather = weather.rename(columns={"time": "local_time", "temperature_2m": "temperature", "relative_humidity_2m": "humidity"})
weather.to_csv(INTERIM / "weather_hourly.csv.gz", index=False)
print(f"linhas horárias: {len(weather):,} | cidades: {weather['city'].nunique()}")

Clima:   0%|          | 0/26 [00:00<?, ?it/s]

linhas horárias: 1,113,216 | cidades: 26


## **7. Exportação e auditoria**

| arquivo | conteúdo |
|---|---|
| `data/interim/cbf_games.csv` | calendário, estádio e placar oficiais de todas as partidas de 2020 a 2026 |
| `data/interim/espn_events.csv` | eventos da ESPN com público, estádio e horário a partir de 2022 |
| `data/interim/venue_cities.csv` | coordenadas de cada cidade-sede |
| `data/interim/weather_hourly.csv.gz` | temperatura, precipitação e umidade por hora em cada cidade |

A auditoria verifica o tamanho de cada temporada concluída e a proporção de eventos da ESPN com público.

In [8]:
season_counts = cbf_games.groupby("season")["game_id"].nunique()
espn_with_attendance = (espn_events["attendance"].fillna(0) > 0).mean()
audit = pd.Series({
    "partidas_cbf": len(cbf_games),
    "partidas_cbf_disputadas_pos_pandemia": int((played["date"].dt.date < TODAY).sum()),
    "eventos_espn": len(espn_events),
    "proporcao_espn_com_publico": round(espn_with_attendance, 3),
    "cidades_sede": len(cities),
    "linhas_de_clima": len(weather),
}, name="valor")
display(audit.to_frame())
assert season_counts.drop(CURRENT_SEASON).eq(380).all(), f"Temporadas concluídas com partidas faltando: {season_counts.to_dict()}"
assert cbf_games["game_id"].is_unique, "game_id da CBF duplicado"
assert espn_with_attendance > 0.8, "Menos de 80% dos eventos da ESPN têm público"
assert cities["latitude"].notna().all(), "Há cidades-sede sem geocodificação"

,valor
partidas_cbf,2660.000
partidas_cbf_disputadas_pos_pandemia,1880.000
eventos_espn,1883.000
proporcao_espn_com_publico,0.891
cidades_sede,26.000
linhas_de_clima,1113216.000
